In [2]:
import json

import torch
import gc
from sentence_transformers import SentenceTransformer
import pandas as pd
import ipywidgets as w
from pathlib import Path

In [14]:
title_text_pairs = pd.read_parquet("articles_10k_test.parquet")[:]

texts = title_text_pairs["text"].tolist()
titles = title_text_pairs["section_title"].tolist()


def truncate_text(text, max_length=200):
    if len(text) > max_length:
        return text[:max_length] + '...'
    return text


def find_best_title(index_in_titles: int, k=5, print_result: bool = True):
    text_embedding = text_embeddings[index_in_titles]
    similarities: torch.Tensor = model.similarity(text_embedding, title_embeddings)

    best_indices = similarities.topk(k=k, largest=True).indices.cpu().numpy().astype(int).flatten()

    output = None

    text = texts[index_in_titles] if index_in_titles < len(texts) else None
    true_title = titles[index_in_titles] if index_in_titles < len(titles) else None
    if true_title:
        ranking = similarities[0].argsort(descending=True)
        title_rank = ranking.tolist().index(index_in_titles) + 1
    else:
        title_rank = None

    output = f"""
Text: \n {text if text else 'No text provided'}\n\n
True title: {true_title}, score: {similarities[0, index_in_titles]:.4f}, title rank: {title_rank if title_rank else "n/a"}\n\n
Top {k} titles for text:
"""
    for i, index in enumerate(best_indices):
        output += f"{i + 1}. {titles[index]} (similarity: {similarities[0, index].item():.4f})\n"

    if print_result:
        print(output)

    return best_indices, similarities.cpu().numpy()[0, best_indices], output


def find_best_text(index_in_titles: int, k=5, print_result: bool = True):
    title_embedding = title_embeddings[index_in_titles]
    similarities: torch.Tensor = model.similarity(title_embedding, text_embeddings)

    best_indices = similarities.topk(k=k, largest=True).indices.cpu().numpy().astype(int).flatten()

    output = None

    title = titles[index_in_titles] if index_in_titles < len(texts) else None
    true_text = texts[index_in_titles] if index_in_titles < len(titles) else None
    if true_text:
        ranking = similarities[0].argsort(descending=True)
        text_rank = ranking.tolist().index(index_in_titles) + 1
    else:
        text_rank = None

    output = f"""
Title: \n {title}\n\n
True text: {true_text},\n\n Score: {similarities[0, index_in_titles]:.4f}, text rank: {text_rank if text_rank else "n/a"}\n\n
Top {k} texts matching title:
"""
    for i, index in enumerate(best_indices):
        output += f"{i + 1}. {truncate_text(texts[index], 100)} (similarity: {similarities[0, index].item():.4f})\n\n"

    if print_result:
        print(output)

    return best_indices, similarities.cpu().numpy()[0, best_indices], output


title_text_pairs

,page_id,page_title,section_title,text
0,31208,Ірпінь,Економіка та промисловість,У вересні 2016 року Ірпінському регіоні діяли ...
1,41207,Віскі,Виробничий процес,Виробничий процес складається з наступних осно...
2,3886,Буддизм,Праджня (мудрість): медитація віпасана,"Праджня означає мудрість, що базується на усві..."
3,25937,Мен (штат),Економіка,"Виробництво: целюлозно-паперова промисловість,..."
4,7353,Малаві,Малаві,Респу́бліка Мала́ві (до 1964 Нья́саленд) — кра...
...,...,...,...,...
9985,19354,Авреліан,Вбивство,У 275 році Авреліан на чолі великої армії руши...
9986,54195,Шахтар (Донецьк),Шахтар (Донецьк),«Шахта́р» — український футбольний клуб з міст...
9987,7790,Авокадо,Отруйність для тварин,"У низці джерел зазначено, що листя дерева авок..."
9988,28505,Рок-музика,Зародження альтернативної музичної культури,Хоча почини Velvet Underground в першій полови...


In [2]:
model = SentenceTransformer(
    # "all-MiniLM-L6-v2"
    # "all-distilroberta-v1"
    # "intfloat/multilingual-e5-small"
    "m-rudko-pn/e5-base-ukr-wikipedia"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

You are trying to use a model that was created with Sentence Transformers version 4.1.0, but you're currently using version 4.0.1. This might cause unexpected behavior or errors. In that case, try to update to the latest version.


README.md:   0%|          | 0.00/59.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [18]:
title_embeddings = model.encode(titles, convert_to_tensor=True)
text_embeddings = model.encode(texts, convert_to_tensor=True)

In [19]:
index_slider = w.IntSlider(
    value=0,
    min=0,
    max=len(titles) - 1,
    description='Index in titles:',
    continuous_update=False,
)

text_display = w.Textarea(
    value='',
    description='Text:',
    layout=w.Layout(width='100%'),
)


def update_text_display(change):
    index = change['new']
    best_indices, similarities, output = find_best_text(index, k=5, print_result=True)
    text_display.value = output


index_slider.observe(update_text_display, names='value')

w.VBox([
    index_slider,
    text_display])

In [44]:
embed_index = 10
find_best_title(embed_index, k=5)


Text: 
 Практично одразу після сходження Якова I на англійський престол почалось спочатку обережне, але таке, що поступово набирало силу, протистояння парламенту й короля Англії. Уже у 1604 році, незважаючи на добровільну відмову короля від своїх прерогатив у сфері встановлення монополій та королівської опіки, парламент Англії не затвердив субсидії королю. У 1605 році королю вдалось домогтись санкціонування екстраординарного податку, однак надходження від нього були недостатніми. Яків I почав стягувати податки з імпортних товарів без згоди парламенту, що спричинило бурю невдоволення останнього. Однак, завдяки митній реформі Роберта Сесіла, королю тимчасово вдалось стабілізувати королівські фінанси. У 1610 році Сесіл запропонував проєкт «Великого контракту»: затвердження парламентом щорічної фіксованої субсидії королю на підставі загального земельного податку в обмін на відмову Якова від королівських феодальних прерогатив. Однак цей проєкт було провалено в англійському парламенті. У ві

(array([54, 55, 96, 60, 44]),
 array([0.55155706, 0.539093  , 0.5137538 , 0.5126001 , 0.50240517],
       dtype=float32),
 '\nText: \n Практично одразу після сходження Якова I на англійський престол почалось спочатку обережне, але таке, що поступово набирало силу, протистояння парламенту й короля Англії. Уже у 1604 році, незважаючи на добровільну відмову короля від своїх прерогатив у сфері встановлення монополій та королівської опіки, парламент Англії не затвердив субсидії королю. У 1605 році королю вдалось домогтись санкціонування екстраординарного податку, однак надходження від нього були недостатніми. Яків I почав стягувати податки з імпортних товарів без згоди парламенту, що спричинило бурю невдоволення останнього. Однак, завдяки митній реформі Роберта Сесіла, королю тимчасово вдалось стабілізувати королівські фінанси. У 1610 році Сесіл запропонував проєкт «Великого контракту»: затвердження парламентом щорічної фіксованої субсидії королю на підставі загального земельного податку в 

In [14]:
model.similarity(
    text_embeddings[0],
    title_embeddings
).cpu().numpy()

array([[ 0.4271259 ],
       [ 0.4602333 ],
       [ 0.3416525 ],
       [ 0.31190777],
       [ 0.27928132],
       [ 0.18429232],
       [ 0.23652565],
       [ 0.32210103],
       [ 0.22147965],
       [ 0.49118572],
       [ 0.34268782],
       [ 0.07309867],
       [ 0.3583073 ],
       [ 0.33524364],
       [ 0.32324874],
       [ 0.10508177],
       [ 0.33704078],
       [ 0.2782976 ],
       [ 0.27491933],
       [ 0.22147965],
       [ 0.18772821],
       [ 0.1695195 ],
       [ 0.15304266],
       [ 0.34330183],
       [ 0.21900168],
       [ 0.35899848],
       [ 0.29017684],
       [ 0.33912545],
       [ 0.39387938],
       [ 0.26040682],
       [ 0.37173188],
       [ 0.33264837],
       [ 0.4058875 ],
       [ 0.13905106],
       [ 0.32794917],
       [ 0.32288396],
       [ 0.3526184 ],
       [ 0.36428007],
       [ 0.27154016],
       [ 0.30646205],
       [ 0.2837062 ],
       [ 0.45366964],
       [ 0.41152763],
       [ 0.12595874],
       [ 0.48087996],
       [ 0

# Training

In [5]:
from datasets import Dataset, Features, Value, ClassLabel

train_df = pd.read_parquet("articles_10k_train.parquet")[["section_title", "text"]].rename(columns={
    "section_title": "anchor",
    "text": "positive"
})

test_df = pd.read_parquet("articles_10k_test.parquet")[["section_title", "text"]].rename(columns={
    "section_title": "anchor",
    "text": "positive"
})

train_dataset = Dataset.from_pandas(train_df)

test_dataset = Dataset.from_pandas(test_df)

In [8]:
model = SentenceTransformer(
    # "all-MiniLM-L6-v2"
    # "all-distilroberta-v1"
    # "intfloat/multilingual-e5-base",
    "intfloat/multilingual-e5-small",
)

In [9]:
gc.collect()
torch.cuda.empty_cache()

In [10]:
from sentence_transformers import losses, SentenceTransformerTrainer, SentenceTransformerTrainingArguments

args = SentenceTransformerTrainingArguments(
    output_dir="output",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=10,
    warmup_steps=100,
    logging_steps=40,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_strategy="steps",
    eval_strategy="steps",
    bf16=True
)

trainer = SentenceTransformerTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    loss=losses.MultipleNegativesRankingLoss(model=model),
    args=args
)

trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss
40,15.743700,0.474158
80,5.946200,0.256417
120,4.383400,0.229983
160,4.260200,0.212394
200,4.103600,0.202519
240,3.706000,0.194975
280,3.914300,0.192795
320,3.735000,0.187499
360,3.697700,0.184524
400,3.581000,0.183982


TrainOutput(global_step=499, training_loss=4.9314644914829655, metrics={'train_runtime': 1155.0109, 'train_samples_per_second': 69.187, 'train_steps_per_second': 0.432, 'total_flos': 0.0, 'train_loss': 4.9314644914829655, 'epoch': 0.998998998998999})

# MTEB eval

In [6]:
from mteb import MTEB
from datasets import load_dataset
from mteb.abstasks import AbsTaskRetrieval
from mteb.abstasks.TaskMetadata import TaskMetadata


class UkrWikiRetrieval(AbsTaskRetrieval):
    metadata = TaskMetadata(
        name="UkrWikiRetrieval",
        dataset={
            "path": "facebook/belebele",
            "revision": "75b399394a9803252cfec289d103de462763db7c",
        },
        description=(
                "Belebele is a multiple-choice machine reading comprehension (MRC) dataset spanning 122 language variants "
                + "(including 115 distinct languages and their scripts)"
        ),
        type="Retrieval",
        category="s2p",
        modalities=["text"],
        eval_splits=["test"],
        eval_langs=["ukr"],
        reference="https://arxiv.org/abs/2308.16884",
        main_score="ndcg_at_10",
        license="cc-by-sa-4.0",
        domains=["Web", "News", "Written"],
        sample_creation="created",  # number of languages * 900
        date=("2023-08-31", "2023-08-31"),
        task_subtypes=["Question answering"],
        annotations_creators="expert-annotated",
        dialect=[],
        bibtex_citation="""@article{bandarkar2023belebele,
    title={The Belebele Benchmark: a Parallel Reading Comprehension Dataset in 122 Language Variants},
    author={Lucas Bandarkar and Davis Liang and Benjamin Muller and Mikel Artetxe and Satya Narayan Shukla and Donald Husa and Naman Goyal and Abhinandan Krishnan and Luke Zettlemoyer and Madian Khabsa},
    year={2023},
    journal={arXiv preprint arXiv:2308.16884}
    }""", )

    def load_data(self, **kwargs):
        dataset = load_dataset("m-rudko-pn/ukrainian-wikipedia-articles", split=kwargs.get("split", "validation"))
        print(dataset)

        corpus = {}
        queries = {}
        relevant_docs = {}

        title_to_ids = {}

        for i, example in enumerate(dataset):
            doc_id = str(example["page_id"])
            title = example["section_title"]
            text = example["text"]

            corpus[doc_id] = {
                "title": "",
                "text": text if text is not None else "",
            }

            if title not in title_to_ids:
                title_to_ids[title] = []
            title_to_ids[title].append(doc_id)

            # queries[doc_id] = title if title is not None else ""
            # relevant_docs[doc_id] = {doc_id: 1}  # Each query should ideally retrieve its own document

        for title, doc_ids in title_to_ids.items():
            for doc_id in doc_ids:
                queries[doc_id] = title if title is not None else ""
                relevant_docs[doc_id] = {rel_id: 1 for rel_id in doc_ids}

        self.corpus = {"test": corpus}
        self.queries = {"test": queries}
        self.relevant_docs = {"test": relevant_docs}

        print(f"Loaded {len(self.corpus['test'])} documents.")


class UkrPravdaRetrieval(AbsTaskRetrieval):
    metadata = TaskMetadata(
        name="UkrPravdaRetrieval",
        dataset={
            "path": "facebook/belebele",
            "revision": "75b399394a9803252cfec289d103de462763db7c",
        },
        description=(
                "Belebele is a multiple-choice machine reading comprehension (MRC) dataset spanning 122 language variants "
                + "(including 115 distinct languages and their scripts)"
        ),
        type="Retrieval",
        category="s2p",
        modalities=["text"],
        eval_splits=["test"],
        eval_langs=["ukr"],
        reference="https://arxiv.org/abs/2308.16884",
        main_score="ndcg_at_10",
        license="cc-by-sa-4.0",
        domains=["Web", "News", "Written"],
        sample_creation="created",  # number of languages * 900
        date=("2023-08-31", "2023-08-31"),
        task_subtypes=["Question answering"],
        annotations_creators="expert-annotated",
        dialect=[],
        bibtex_citation="""@article{bandarkar2023belebele,
    title={The Belebele Benchmark: a Parallel Reading Comprehension Dataset in 122 Language Variants},
    author={Lucas Bandarkar and Davis Liang and Benjamin Muller and Mikel Artetxe and Satya Narayan Shukla and Donald Husa and Naman Goyal and Abhinandan Krishnan and Luke Zettlemoyer and Madian Khabsa},
    year={2023},
    journal={arXiv preprint arXiv:2308.16884}
    }""", )

    def load_data(self, **kwargs):
        dataset = load_dataset("shamotskyi/ukr_pravda_2y", split="train[:5%]")
        print(dataset)

        corpus = {}
        queries = {}
        relevant_docs = {}

        for i, example in enumerate(dataset):
            doc_id = str(example["art_id"])
            title = example["ukr_title"]
            text = example["ukr_text"]

            corpus[doc_id] = {
                "title": "",
                "text": text if text is not None else "",
            }

            queries[doc_id] = title if title is not None else ""
            relevant_docs[doc_id] = {doc_id: 1}  # Each query should ideally retrieve its own document

        self.corpus = {"test": corpus}
        self.queries = {"test": queries}
        self.relevant_docs = {"test": relevant_docs}

        print(f"Loaded {len(self.corpus['test'])} documents.")


In [13]:
from mteb import MTEB
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base")
# model = SentenceTransformer("youscan/ukr-roberta-base")
# model = SentenceTransformer("m-rudko-pn/e5-small-ukr-wikipedia")
# model = SentenceTransformer("output/checkpoint-499")
benchmark = MTEB(tasks=[UkrWikiRetrieval(), UkrPravdaRetrieval()])
res = benchmark.run(model, verbosity=2, output_folder="results/wiki", overwrite_results=False,
                    encode_kwargs={"batch_size": 64})
res

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Retrieval

- UkrWikiRetrieval, s2p

- UkrPravdaRetrieval, s2p

Overwrite dataset info from restored data version if exists.
INFO:datasets.builder:Overwrite dataset info from restored data version if exists.
Loading Dataset info from /home/imax/.cache/huggingface/datasets/m-rudko-pn___ukrainian-wikipedia-articles/default/0.0.0/2f6a643539579c8a1f8358b71cb0e5f3c57673cc
INFO:datasets.info:Loading Dataset info from /home/imax/.cache/huggingface/datasets/m-rudko-pn___ukrainian-wikipedia-articles/default/0.0.0/2f6a643539579c8a1f8358b71cb0e5f3c57673cc
Found cached dataset ukrainian-wikipedia-articles (/home/imax/.cache/huggingface/datasets/m-rudko-pn___ukrainian-wikipedia-articles/default/0.0.0/2f6a643539579c8a1f8358b71cb0e5f3c57673cc)
INFO:datasets.builder:Found cached dataset ukrainian-wikipedia-articles (/home/imax/.cache/huggingface/datasets/m-rudko-pn___ukrainian-wikipedia-articles/default/0.0.0/2f6a643539579c8a1f8358b71cb0e5f3c57673cc)
Loading Dataset info from /home/imax/.cache/huggingface/datasets/m-rudko-pn___ukrainian-wikipedia-articles/default/

...

Overwrite dataset info from restored data version if exists.
INFO:datasets.builder:Overwrite dataset info from restored data version if exists.
Loading Dataset info from /home/imax/.cache/huggingface/datasets/shamotskyi___ukr_pravda_2y/default/0.0.0/be2be302c4c659d362b6fae64d7526cd1901bca6
INFO:datasets.info:Loading Dataset info from /home/imax/.cache/huggingface/datasets/shamotskyi___ukr_pravda_2y/default/0.0.0/be2be302c4c659d362b6fae64d7526cd1901bca6
Found cached dataset ukr_pravda_2y (/home/imax/.cache/huggingface/datasets/shamotskyi___ukr_pravda_2y/default/0.0.0/be2be302c4c659d362b6fae64d7526cd1901bca6)
INFO:datasets.builder:Found cached dataset ukr_pravda_2y (/home/imax/.cache/huggingface/datasets/shamotskyi___ukr_pravda_2y/default/0.0.0/be2be302c4c659d362b6fae64d7526cd1901bca6)
Loading Dataset info from /home/imax/.cache/huggingface/datasets/shamotskyi___ukr_pravda_2y/default/0.0.0/be2be302c4c659d362b6fae64d7526cd1901bca6
INFO:datasets.info:Loading Dataset info from /home/imax/.c

Dataset({
    features: ['art_id', 'date_published', 'tags', 'ukr_uri', 'ukr_title', 'ukr_author_name', 'ukr_text', 'ukr_tags', 'ukr_tags_full', 'rus_uri', 'rus_title', 'rus_author_name', 'rus_text', 'rus_tags', 'rus_tags_full', 'eng_uri', 'eng_title', 'eng_author_name', 'eng_text', 'eng_tags', 'eng_tags_full'],
    num_rows: 3081
})
Loaded 3081 documents.


Batches:   0%|          | 0/49 [00:00<?, ?it/s]

Batches:   0%|          | 0/49 [00:00<?, ?it/s]

[TaskResult(task_name=UkrWikiRetrieval, scores=...),
 TaskResult(task_name=UkrPravdaRetrieval, scores=...)]

In [60]:
import json

result_root = Path('/home/imax/AllHomework/Golden-Retriever/results/wiki')


def load_results() -> pd.DataFrame:
    results = {}
    for model in result_root.glob('*'):
        for model_result in model.glob('**/*.json'):
            if model_result.stem == 'model_meta':
                continue

            if not model_result.is_file():
                continue

            model_name = model.stem
            if model_name not in results:
                results[model_name] = {}
            with open(model_result, 'r') as json_file:
                results[model_name][model_result.stem] = json.load(json_file)

    result_index = pd.MultiIndex.from_product([['UkrWikiRetrieval', 'UkrPravdaRetrieval'],
                                               ['recall_at_1', 'recall_at_3', 'recall_at_5', 'ndcg_at_1', 'ndcg_at_3',
                                                'ndcg_at_5', ]], names=['task', 'score'])

    df = pd.DataFrame(results).T.reset_index(names=['model_name'])

    result_df = pd.DataFrame(columns=result_index, index=df['model_name']).reset_index(names=['model_name'])

    for dataset in result_index.levels[0]:
        dataset_result_df = pd.json_normalize((pd.json_normalize(df[dataset])['scores.test']).explode())

        for score in result_index.levels[1]:
            result_df[(dataset, score)] = dataset_result_df[score].values
    return result_df.set_index('model_name')

In [61]:
load_results()

task                               UkrWikiRetrieval                          \
score                                   recall_at_1 recall_at_3 recall_at_5   
model_name                                                                    
no_model_name_available                     0.34950     0.42885     0.47230   
m-rudko-pn__e5-base-ukr-wikipedia           0.38886     0.48781     0.53003   
m-rudko-pn__e5-small-ukr-wikipedia          0.34950     0.42885     0.47230   
youscan__ukr-roberta-base                   0.02342     0.04146     0.05266   
intfloat__multilingual-e5-small             0.24394     0.29851     0.32812   
intfloat__multilingual-e5-base              0.25816     0.31544     0.33937   

task                                                              \
score                              ndcg_at_1 ndcg_at_3 ndcg_at_5   
model_name                                                         
no_model_name_available              0.47865   0.49317   0.51018   
m-rudko-pn__e5-base-ukr-wikipedia    0.55346   0.57362   0.57936   
m-rudko-pn__e5-small-ukr-wikipedia   0.47865   0.49317   0.51018   
youscan__ukr-roberta-base            0.03983   0.04760   0.04859   
intfloat__multilingual-e5-small      0.26337   0.28951   0.30110   
intfloat__multilingual-e5-base       0.27915   0.30857   0.31753   

task                               UkrPravdaRetrieval                          \
score                                     recall_at_1 recall_at_3 recall_at_5   
model_name                                                                      
no_model_name_available                           NaN         NaN         NaN   
m-rudko-pn__e5-base-ukr-wikipedia             0.75105     0.83836     0.85557   
m-rudko-pn__e5-small-ukr-wikipedia            0.80493     0.87634     0.89159   
youscan__ukr-roberta-base                     0.13859     0.19474     0.22655   
intfloat__multilingual-e5-small               0.79617     0.87374     0.89387   
intfloat__multilingual-e5-base                0.78351     0.85881     0.87634   

task                                                              
score                              ndcg_at_1 ndcg_at_3 ndcg_at_5  
model_name                                                        
no_model_name_available                  NaN       NaN       NaN  
m-rudko-pn__e5-base-ukr-wikipedia    0.75105   0.80274   0.80987  
m-rudko-pn__e5-small-ukr-wikipedia   0.80493   0.84773   0.85400  
youscan__ukr-roberta-base            0.13859   0.17168   0.18484  
intfloat__multilingual-e5-small      0.79617   0.84261   0.85096  
intfloat__multilingual-e5-base       0.78351   0.82856   0.83576

In [30]:
df = pd.DataFrame(load_results()).T.reset_index(names=['model_name'])
# df.explode('no_model_name_available')
# df

In [50]:
pd.json_normalize((pd.json_normalize(df['UkrWikiRetrieval'])['scores.test']).explode())

,ndcg_at_1,ndcg_at_3,ndcg_at_5,ndcg_at_10,ndcg_at_20,ndcg_at_100,ndcg_at_1000,map_at_1,map_at_3,map_at_5,...,nauc_mrr_at_20_diff1,nauc_mrr_at_100_max,nauc_mrr_at_100_std,nauc_mrr_at_100_diff1,nauc_mrr_at_1000_max,nauc_mrr_at_1000_std,nauc_mrr_at_1000_diff1,main_score,hf_subset,languages
0,0.47865,0.49317,0.51018,0.52066,0.53175,0.55499,0.58656,0.34950,0.38715,0.40032,...,0.596645,0.429165,-0.110409,0.596821,0.428948,-0.110776,0.596966,0.52066,default,[ukr]
1,0.55346,0.57362,0.57936,0.58869,0.60085,0.61699,0.64085,0.38886,0.43889,0.45297,...,0.547723,0.443914,-0.107706,0.548028,0.443801,-0.107886,0.548127,0.58869,default,[ukr]
2,0.47865,0.49317,0.51018,0.52066,0.53175,0.55499,0.58656,0.34950,0.38715,0.40032,...,0.596645,0.429165,-0.110409,0.596821,0.428948,-0.110776,0.596966,0.52066,default,[ukr]
3,0.03983,0.04760,0.04859,0.05250,0.05663,0.07487,0.12707,0.02342,0.03125,0.03380,...,0.344079,0.329647,0.153241,0.334441,0.329399,0.154398,0.333989,0.05250,default,[ukr]
4,0.26337,0.28951,0.30110,0.31578,0.32807,0.35573,0.40864,0.24394,0.26843,0.27571,...,0.647659,0.418163,0.035519,0.646566,0.418280,0.035482,0.646947,0.31578,default,[ukr]
5,0.27915,0.30857,0.31753,0.32564,0.33698,0.36314,0.41718,0.25816,0.28403,0.29007,...,0.630770,0.417591,-0.114511,0.629509,0.417935,-0.114647,0.629974,0.32564,default,[ukr]


In [41]:
result_index = pd.MultiIndex.from_product([['UkrWikiRetrieval', 'UkrPravdaRetrieval'],
                                           ['recall_at_1', 'recall_at_3', 'recall_at_5', 'ndcg_at_1', 'ndcg_at_3',
                                            'ndcg_at_5', ]], names=['task', 'score'])

df[('UkrWikiRetrieval', 'scores')] = pd.json_normalize(df['UkrWikiRetrieval'])['scores.test']

df

,model_name,UkrWikiRetrieval,UkrPravdaRetrieval,"(UkrWikiRetrieval, scores)"
0,no_model_name_available,{'dataset_revision': '75b399394a9803252cfec289...,NaN,"[{'ndcg_at_1': 0.47865, 'ndcg_at_3': 0.49317, ..."
1,m-rudko-pn__e5-base-ukr-wikipedia,{'dataset_revision': '75b399394a9803252cfec289...,{'dataset_revision': '75b399394a9803252cfec289...,"[{'ndcg_at_1': 0.55346, 'ndcg_at_3': 0.57362, ..."
2,m-rudko-pn__e5-small-ukr-wikipedia,{'dataset_revision': '75b399394a9803252cfec289...,{'dataset_revision': '75b399394a9803252cfec289...,"[{'ndcg_at_1': 0.47865, 'ndcg_at_3': 0.49317, ..."
3,youscan__ukr-roberta-base,{'dataset_revision': '75b399394a9803252cfec289...,{'dataset_revision': '75b399394a9803252cfec289...,"[{'ndcg_at_1': 0.03983, 'ndcg_at_3': 0.0476, '..."
4,intfloat__multilingual-e5-small,{'dataset_revision': '75b399394a9803252cfec289...,{'dataset_revision': '75b399394a9803252cfec289...,"[{'ndcg_at_1': 0.26337, 'ndcg_at_3': 0.28951, ..."
5,intfloat__multilingual-e5-base,{'dataset_revision': '75b399394a9803252cfec289...,{'dataset_revision': '75b399394a9803252cfec289...,"[{'ndcg_at_1': 0.27915, 'ndcg_at_3': 0.30857, ..."


In [57]:
result_df = pd.DataFrame(columns=result_index, index=df['model_name']).reset_index(names=['model_name'])

for dataset in result_index.levels[0]:
    dataset_result_df = pd.json_normalize((pd.json_normalize(df[dataset])['scores.test']).explode())

    for score in result_index.levels[1]:
        result_df[(dataset, score)] = dataset_result_df[score].values

print(result_df.to_markdown())

|    | ('model_name', '')                 |   ('UkrWikiRetrieval', 'recall_at_1') |   ('UkrWikiRetrieval', 'recall_at_3') |   ('UkrWikiRetrieval', 'recall_at_5') |   ('UkrWikiRetrieval', 'ndcg_at_1') |   ('UkrWikiRetrieval', 'ndcg_at_3') |   ('UkrWikiRetrieval', 'ndcg_at_5') |   ('UkrPravdaRetrieval', 'recall_at_1') |   ('UkrPravdaRetrieval', 'recall_at_3') |   ('UkrPravdaRetrieval', 'recall_at_5') |   ('UkrPravdaRetrieval', 'ndcg_at_1') |   ('UkrPravdaRetrieval', 'ndcg_at_3') |   ('UkrPravdaRetrieval', 'ndcg_at_5') |
|---:|:-----------------------------------|--------------------------------------:|--------------------------------------:|--------------------------------------:|------------------------------------:|------------------------------------:|------------------------------------:|----------------------------------------:|----------------------------------------:|----------------------------------------:|--------------------------------------:|---------------------------------

In [54]:
pd.json_normalize((pd.json_normalize(df['UkrWikiRetrieval'])['scores.test']).explode())

,ndcg_at_1,ndcg_at_3,ndcg_at_5,ndcg_at_10,ndcg_at_20,ndcg_at_100,ndcg_at_1000,map_at_1,map_at_3,map_at_5,...,nauc_mrr_at_20_diff1,nauc_mrr_at_100_max,nauc_mrr_at_100_std,nauc_mrr_at_100_diff1,nauc_mrr_at_1000_max,nauc_mrr_at_1000_std,nauc_mrr_at_1000_diff1,main_score,hf_subset,languages
0,0.47865,0.49317,0.51018,0.52066,0.53175,0.55499,0.58656,0.34950,0.38715,0.40032,...,0.596645,0.429165,-0.110409,0.596821,0.428948,-0.110776,0.596966,0.52066,default,[ukr]
1,0.55346,0.57362,0.57936,0.58869,0.60085,0.61699,0.64085,0.38886,0.43889,0.45297,...,0.547723,0.443914,-0.107706,0.548028,0.443801,-0.107886,0.548127,0.58869,default,[ukr]
2,0.47865,0.49317,0.51018,0.52066,0.53175,0.55499,0.58656,0.34950,0.38715,0.40032,...,0.596645,0.429165,-0.110409,0.596821,0.428948,-0.110776,0.596966,0.52066,default,[ukr]
3,0.03983,0.04760,0.04859,0.05250,0.05663,0.07487,0.12707,0.02342,0.03125,0.03380,...,0.344079,0.329647,0.153241,0.334441,0.329399,0.154398,0.333989,0.05250,default,[ukr]
4,0.26337,0.28951,0.30110,0.31578,0.32807,0.35573,0.40864,0.24394,0.26843,0.27571,...,0.647659,0.418163,0.035519,0.646566,0.418280,0.035482,0.646947,0.31578,default,[ukr]
5,0.27915,0.30857,0.31753,0.32564,0.33698,0.36314,0.41718,0.25816,0.28403,0.29007,...,0.630770,0.417591,-0.114511,0.629509,0.417935,-0.114647,0.629974,0.32564,default,[ukr]
